# Repair 7 failed DeepSeek predictions

Retry đúng các case generation bị lỗi, dùng lại pipeline FAISS + 10 BM25 shards + graph, sau đó xuất repaired run đủ predictions, metrics, report và README. Artifact gốc không bị ghi đè.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, random, shutil, subprocess, sys, time

REPO_URL = 'https://github.com/PhuongThao-2005/TextMining.git'
REPO_BRANCH = 'my'  # branch có runner hỗ trợ BM25 shards
REPO_DIR = Path('/kaggle/working/TextMining_retry_runner_my')
ORIGINAL_RUN_DIR = Path('/kaggle/working/TextMining_retry_runner_main/e2e_LLM_Reasoning/deepseekv3.1-thinking-CoT')

BENCHMARK_SOURCE = Path('/kaggle/input/datasets/phuongthao205/qa-legalrag/Benchmark/qa_final.jsonl')
CORPUS_SOURCE = Path('/kaggle/input/datasets/nguyenlethienlyy/text-mining-data-preprocessed/documents.jsonl')
FAISS_INDEX_SOURCE = Path('/kaggle/input/datasets/kittrntunk/faiss-chunk-meta/index.faiss')
FAISS_PAYLOADS_SOURCE = Path('/kaggle/input/datasets/kittrntunk/faiss-chunk-meta/payloads.jsonl')
GRAPH_SOURCE = Path('/kaggle/input/datasets/myvnthdim/kg-pkl/knowledge_graph.gpickle')
BM25_SHARDS_SOURCE = Path('/kaggle/input/datasets/nguyenlethienlyy/bm25-tokenized/bm25')
BM25_EXPECTED_SHARDS = 10
RUNS_ROOT = Path('/kaggle/working/evaluation_runs/ablation')
EXPECTED_FAILED_CASES = 7
RETRY_RUN_ID = 'deepseek_retry_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
FAISS_RUNTIME_DIR = Path('/kaggle/working/e2e_retry_runtime') / RETRY_RUN_ID / 'faiss'


In [ ]:
if not (REPO_DIR / '.git').is_dir():
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a Git repository.')
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
branch = subprocess.run(['git', '-C', str(REPO_DIR), 'branch', '--show-current'], check=True, capture_output=True, text=True).stdout.strip()
runner_path = REPO_DIR / 'scripts' / 'run_ablation_config.py'
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
assert runner_path.is_file(), f'Missing sharded-BM25 runner: {runner_path}'
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))
for name in tuple(sys.modules):
    if name == 'scripts' or name.startswith('scripts.') or name in {'evaluation', 'generation', 'retrieval'}:
        sys.modules.pop(name, None)
print({'branch': branch, 'runner': str(runner_path)})


In [ ]:
# Load Kaggle Secrets when they are not already present in the environment.
from kaggle_secrets import UserSecretsClient
secret_client = UserSecretsClient()
for secret_name in ('LLM_API_KEY', 'LLM_BASE_URL'):
    if not os.environ.get(secret_name):
        os.environ[secret_name] = secret_client.get_secret(secret_name)
assert os.environ.get('LLM_API_KEY')
assert os.environ.get('LLM_BASE_URL')
print('LLM credentials loaded (values hidden).')


In [ ]:
required_files = [BENCHMARK_SOURCE, CORPUS_SOURCE, FAISS_INDEX_SOURCE, FAISS_PAYLOADS_SOURCE, GRAPH_SOURCE]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing Kaggle input files: ' + '; '.join(missing))
if not ORIGINAL_RUN_DIR.is_dir():
    raise FileNotFoundError(f'Original run missing: {ORIGINAL_RUN_DIR}')
required_artifacts = ['manifest.json', 'resolved_config.yaml', 'e2e_predictions.jsonl', 'e2e_metrics.json', 'errors.jsonl']
missing_artifacts = [name for name in required_artifacts if not (ORIGINAL_RUN_DIR / name).is_file()]
if missing_artifacts:
    raise FileNotFoundError(f'Missing original artifacts: {missing_artifacts}')
shards = sorted(path for path in BM25_SHARDS_SOURCE.glob('shard_*') if path.is_dir())
if len(shards) != BM25_EXPECTED_SHARDS:
    raise RuntimeError(f'Expected {BM25_EXPECTED_SHARDS} BM25 shards, found {len(shards)}')
for shard in shards:
    for name in ('bm25_index.pkl', 'bm25_metadata.pkl'):
        if not (shard / name).is_file():
            raise FileNotFoundError(f'Missing {name} in {shard}')
FAISS_RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
for target, source in ((FAISS_RUNTIME_DIR / 'index.faiss', FAISS_INDEX_SOURCE), (FAISS_RUNTIME_DIR / 'payloads.jsonl', FAISS_PAYLOADS_SOURCE)):
    if target.exists() or target.is_symlink():
        if target.resolve() != source.resolve():
            raise RuntimeError(f'Unexpected existing runtime artifact: {target}')
    else:
        target.symlink_to(source)
print('All benchmark, FAISS, graph and BM25 inputs are available.')


In [ ]:
import yaml
from collections import Counter

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]
def case_id(row):
    return row.get('qa_id') or row.get('case_id') or row.get('id')

manifest = json.loads((ORIGINAL_RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
base_predictions = read_jsonl(ORIGINAL_RUN_DIR / 'e2e_predictions.jsonl')
errors = read_jsonl(ORIGINAL_RUN_DIR / 'errors.jsonl')
failed_ids = {case_id(row) for row in base_predictions if row.get('status') == 'failed'}
failed_ids.discard(None)
assert len(failed_ids) == EXPECTED_FAILED_CASES, f'Expected {EXPECTED_FAILED_CASES}, found {len(failed_ids)}'
print('Failed IDs:', sorted(failed_ids))
print('Error groups:', Counter((row.get('stage'), row.get('exception_type')) for row in errors))

qa_rows = read_jsonl(BENCHMARK_SOURCE)
retry_rows = [row for row in qa_rows if case_id(row) in failed_ids]
assert len(retry_rows) == len(failed_ids)
retry_qa_path = Path('/kaggle/working') / f'{RETRY_RUN_ID}_benchmark.jsonl'
retry_qa_path.write_text('\n'.join(json.dumps(row, ensure_ascii=False) for row in retry_rows) + '\n', encoding='utf-8')

resolved_config = yaml.safe_load((ORIGINAL_RUN_DIR / 'resolved_config.yaml').read_text(encoding='utf-8'))
resolved_config['benchmark']['path'] = str(retry_qa_path)
resolved_config['corpus']['path'] = str(CORPUS_SOURCE)
dense = resolved_config['retrieval']['dense']
dense['index_path'] = str(FAISS_RUNTIME_DIR)
dense['index_file'] = str(FAISS_RUNTIME_DIR / 'index.faiss')
dense['payloads_path'] = str(FAISS_RUNTIME_DIR / 'payloads.jsonl')
dense.pop('manifest_path', None)
sparse = resolved_config['retrieval'].setdefault('sparse', {})
if sparse.get('enabled'):
    sparse['index_path'] = str(BM25_SHARDS_SOURCE)
    sparse['layout'] = 'sharded'
graph = resolved_config['retrieval'].setdefault('graph', {})
if graph.get('enabled'):
    graph['path'] = str(GRAPH_SOURCE)
resolved_config['output']['root'] = str(RUNS_ROOT)
resolved_config['generation']['max_retries'] = 6
resolved_config['generation']['timeout_seconds'] = 90.0
print('Retry benchmark:', retry_qa_path)
print('Model:', resolved_config['generation']['model'])


In [ ]:
from scripts.run_ablation_config import run_ablation_config
from generation.reasoning_client import GeneratorClient

# Branch my has sharded BM25; add bounded backoff around its generator for 429/5xx failures.
_original_generate = GeneratorClient.generate
def _generate_with_backoff(self, prompt, **kwargs):
    retries = int(kwargs.pop('max_retries', 6))
    for attempt in range(retries + 1):
        try:
            return _original_generate(self, prompt, max_retries=0, **kwargs)
        except RuntimeError as exc:
            message = str(exc).lower()
            transient = any(token in message for token in ('429', 'rate limit', '500', '502', '503', '504', 'timeout'))
            if not transient or attempt >= retries:
                raise
            delay = min(60.0, 2.0 ** (attempt + 1)) + random.uniform(0.0, 1.0)
            print(f'Generator transient error; retry {attempt + 1}/{retries} after {delay:.1f}s')
            time.sleep(delay)
GeneratorClient.generate = _generate_with_backoff

retry_outcome = run_ablation_config(
    str(manifest['config_name']),
    config_file=REPO_DIR / 'configs' / 'ablation_configs.yaml',
    output_root=RUNS_ROOT, run_id=RETRY_RUN_ID, limit=None, dry_run=False,
    project_root=REPO_DIR, resolved_config_override=resolved_config,
)
print({'status': retry_outcome.status, 'output': str(retry_outcome.output_dir), 'error': retry_outcome.error})
if retry_outcome.status != 'completed':
    raise RuntimeError(retry_outcome.error or retry_outcome.status)


In [ ]:
from evaluation.e2e_runner import E2ERunResult, METRIC_KEYS, aggregate_agent_metrics, aggregate_latency, write_e2e_artifacts
from evaluation.metrics import aggregate, aggregate_by
from generation.citations import aggregate_citation_metrics

retry_run_dir = Path(retry_outcome.output_dir)
retry_predictions = read_jsonl(retry_run_dir / 'e2e_predictions.jsonl')
retry_by_id = {case_id(row): row for row in retry_predictions}
unrecovered = [item for item in sorted(failed_ids) if retry_by_id.get(item, {}).get('status') != 'success']
assert not unrecovered, f'Retry still failed: {unrecovered}'
merged = [retry_by_id[case_id(row)] if case_id(row) in failed_ids else row for row in base_predictions]
assert len(merged) == 500 and all(row.get('status') == 'success' for row in merged)

metric_keys = list(METRIC_KEYS)
if merged and 'judge_correctness' in merged[0]:
    metric_keys += ['judge_correctness', 'judge_faithfulness', 'judge_answer_relevancy']
counts = {'total_input': len(merged), 'successful': len(merged), 'failed': 0, 'skipped': 0, 'evaluated': len(merged)}
old_metrics = json.loads((ORIGINAL_RUN_DIR / 'e2e_metrics.json').read_text(encoding='utf-8'))
metrics = {
    'qa_path': str(BENCHMARK_SOURCE), 'config': old_metrics.get('config') or resolved_config, 'counts': counts,
    'metric_denominator': 'Per-metric: averages exclude successful cases where that metric is not applicable.',
    'overall': aggregate(merged, metric_keys),
    'by_category': aggregate_by(merged, 'category', metric_keys),
    'by_answer_type': aggregate_by(merged, 'answer_type', metric_keys),
    'by_difficulty': aggregate_by(merged, 'difficulty', metric_keys),
    'metric_keys': metric_keys, 'agent_metrics': aggregate_agent_metrics(merged),
    'citation_metrics': aggregate_citation_metrics(merged),
}
latency = aggregate_latency(merged)
latency.update({'denominator': 'Cases with a recorded value for each stage.', 'counts': counts})

repaired_dir = Path('/kaggle/working/repaired_ablation_runs') / f"{manifest['run_id']}_repaired"
repaired_dir.mkdir(parents=True, exist_ok=False)
artifacts = write_e2e_artifacts(repaired_dir, E2ERunResult(merged, [], metrics, latency), report_name='report.md')
shutil.copy2(ORIGINAL_RUN_DIR / 'resolved_config.yaml', repaired_dir / 'resolved_config.yaml')
new_manifest = dict(manifest)
new_manifest.update({
    'run_id': f"{manifest['run_id']}_repaired", 'status': 'completed',
    'end_time': datetime.now(timezone.utc).isoformat(), 'output_directory': str(repaired_dir),
    'completed_case_count': 500, 'failed_case_count': 0, 'skipped_case_count': 0,
    'evaluated_case_count': 500, 'error_summary': None,
    'repair': {'original_run': str(ORIGINAL_RUN_DIR), 'retry_run': str(retry_run_dir), 'recovered_case_ids': sorted(failed_ids)},
    'output_artifacts': {**artifacts, 'manifest': str(repaired_dir / 'manifest.json'), 'resolved_config': str(repaired_dir / 'resolved_config.yaml'), 'readme': str(repaired_dir / 'README.md')},
})
(repaired_dir / 'manifest.json').write_text(json.dumps(new_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
lines = ['# Repaired DeepSeek Ablation Result', '', '- Successful: 500', '- Failed: 0', '- Skipped: 0', f'- Recovered cases: {len(failed_ids)}', '', '## Updated metrics', '', '| Metric | Value |', '| --- | ---: |']
for key in metric_keys:
    lines.append(f"| {key} | {float(metrics['overall'].get(key, 0.0)):.4f} |")
(repaired_dir / 'README.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')
archive = shutil.make_archive(str(repaired_dir), 'zip', root_dir=repaired_dir)
print({'repaired_dir': str(repaired_dir), 'archive': archive, 'counts': counts})
